In [71]:
import numpy as np
import altair as alt
import pandas as pd

In [72]:
df: pd.DataFrame = pd.read_csv("student_performance/student_performance.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 714 entries, 0 to 713
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          714 non-null    int64  
 1   school      714 non-null    str    
 2   sex         714 non-null    str    
 3   age         710 non-null    float64
 4   address     712 non-null    str    
 5   famsize     709 non-null    str    
 6   Pstatus     707 non-null    str    
 7   Medu        709 non-null    float64
 8   Fedu        708 non-null    float64
 9   Mjob        713 non-null    str    
 10  Fjob        711 non-null    str    
 11  reason      710 non-null    str    
 12  guardian    711 non-null    str    
 13  traveltime  712 non-null    float64
 14  studytime   710 non-null    float64
 15  failures    711 non-null    float64
 16  schoolsup   712 non-null    str    
 17  famsup      711 non-null    str    
 18  paid        710 non-null    str    
 19  activities  707 non-null    str    
 20 

In [73]:
# Get numeric columns with missing values
columns = df.select_dtypes(include=[np.number]).isnull().sum().where(lambda x: x > 0).dropna().index.tolist()
print("Numeric columns with missing values:", columns)

for col in columns:
    df[col] = df[col].fillna(df[col].median())

# Get categorical columns with missing values
cat_columns = df.select_dtypes(include=['str']).isnull().sum().where(lambda x: x > 0).dropna().index.tolist()
for col in cat_columns:
    df[col] = df[col].fillna(df[col].mode()[0])


Numeric columns with missing values: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'goout', 'Dalc', 'Walc', 'health', 'absences']


In [74]:
df["grade_avrg"] = df[["G3"]]

In [75]:
# 1. Define the brush selection 
brush = alt.selection_interval()

# --- DEFINE CUSTOM COLOR SCALE ---
sex_color_scale = alt.Scale(domain=['F', 'M'], range=['#ff7f0e', '#1f77b4'])

# Base chart for common properties
base = alt.Chart(df).properties(
    width=300,
    height=250
)

# --- TOP LEFT: Absences vs Final Grade (The Anchor Filter Plot) ---
scatter1 = base.mark_circle().encode(
    x=alt.X('G3:Q', title='Final Grade', scale=alt.Scale(domain=[-1, max(df["G3"]) + 1])),
    y=alt.Y('absences:Q', title='Absences', scale=alt.Scale(domain=[-2, max(df["absences"]) + 2])), 
    size=alt.Size('count():Q', title='Number of Students', scale=alt.Scale(range=[20, 600])), 
    color=alt.condition(brush, alt.value('#2c7fb8'), alt.value('lightgray')), 
    tooltip=['G3', 'absences', 'count()']
).add_params(
    brush
).properties(
    title="Final Grade vs. Absences (Drag to Filter!)"
)

study_labels= "datum.value == 1 ? '1: <2 hours' : datum.value == 2 ? '2: 2-5 hours' : datum.value == 3 ? '3: 5-10 hours' : '4: >10 hours'"

# --- TOP RIGHT: Study Time vs Final Grade ---
scatter2 = base.mark_boxplot(extent='min-max').transform_filter(
    brush
).encode(
    x=alt.X('studytime:O', title='Study Time Category', scale=alt.Scale(domain=[1, 2, 3, 4]), axis=alt.Axis(labelExpr=study_labels, labelAngle=-45)),
    y=alt.Y('G3:Q', title='Final Grade', scale=alt.Scale(domain=[0, 20])),
    color=alt.Color('sex:N', title='Sex', scale=sex_color_scale), # Applied flipped colors
    xOffset=alt.XOffset('sex:N', sort=['F', 'M']) # Ensure offset order remains constant
).properties(
    title="Grade Distribution by Study Time"
)


# --- BOTTOM LEFT: Heatmap for Parental Education ---
edu_labels = "datum.value == 0 ? '0: None' : datum.value == 1 ? '1: Primary' : datum.value == 2 ? '2: 5th-9th' : datum.value == 3 ? '3: Secondary' : '4: Higher'"

heatmap_base = base.transform_filter(
    brush
).transform_aggregate(
    count='count()',
    groupby=['Medu', 'Fedu']
).encode(
    # Locked domain 0-4
    x=alt.X('Medu:O', title='Mother Education', scale=alt.Scale(domain=[0, 1, 2, 3, 4]), axis=alt.Axis(labelExpr=edu_labels, labelAngle=-45)),
    # Locked domain 4-0
    y=alt.Y('Fedu:O', title='Father Education', scale=alt.Scale(domain=[4, 3, 2, 1, 0]), axis=alt.Axis(labelExpr=edu_labels))
)

heatmap_rect = heatmap_base.mark_rect().encode(
    color=alt.Color('count:Q', scale=alt.Scale(scheme='viridis'), title='Count')
)

heatmap_text = heatmap_base.mark_text().encode(
    text='count:Q',
    color=alt.condition(
        alt.datum.count > 50,  
        alt.value('black'),
        alt.value('white')
    )
)

heatmap = (heatmap_rect + heatmap_text).properties(
    title="Parental Education Heatmap"
)


# --- BOTTOM RIGHT: Social Life Averages ---
social_plot = alt.Chart(df).transform_filter(
    brush
).transform_fold(
    ['Dalc', 'Walc', 'freetime', 'goout'],
    as_=['Metric', 'Score']
).mark_bar().encode(
    # Locked the Metric domain so columns don't vanish if unselected
    x=alt.X('Metric:N', title='Social Metric', scale=alt.Scale(domain=['Dalc', 'Walc', 'freetime', 'goout']), axis=alt.Axis(labelAngle=0)),
    y=alt.Y('mean(Score):Q', title='Average Score (1=Low, 5=High)', scale=alt.Scale(domain=[0, 5])),
    color=alt.Color('sex:N', title='Sex', scale=sex_color_scale, legend=None), # Applied flipped colors
    # Hardcoded the domain for xOffset. This prevents the "single datapoint" bar stretching bug!
    xOffset=alt.XOffset('sex:N', scale=alt.Scale(domain=['F', 'M'])),
    tooltip=['Metric:N', 'sex:N', 'mean(Score):Q']
).properties(
    width=300,
    height=250,
    title='Social Life Averages'
)

# Use a 2-column grid layout
dashboard = alt.concat(
    scatter1, scatter2, heatmap, social_plot, 
    columns=2
)

dashboard

alt.ConcatChart(...)

In [76]:
#dashboard.save('student_performance_dashboard.html')